In [1]:
import os
import qsprpred

from qsprpred.data import QSPRDataset, RandomSplit
from qsprpred.data.descriptors.fingerprints import MorganFP
import pandas as pd

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [2]:
os.makedirs("dataset_outputs/HIV/data", exist_ok=True)

# Create dataset
dataset = QSPRDataset.fromTableFile(
    filename="data/src/datasets/hiv/data/HIV.csv",
    store_dir="dataset_outputs/HIV/data",
    name="HIVDataset",
    target_props=[{"name": "Y", "task": "SINGLECLASS", "th": [0.5]}],
    random_state=42,
    smiles_col = 'Drug',
    sep=','
)

In [3]:
dataset.getDF()

,Y,Drug,QSPRID,Y_original
QSPRID,,,,
HIVDataset_00000,False,CCC1=[O+][Cu-3]2([O+]=C(CC)C1)[O+]=C(CC)CC(CC)...,HIVDataset_00000,0
HIVDataset_00001,False,C(=Cc1ccccc1)C1=[O+][Cu-3]2([O+]=C(C=Cc3ccccc3...,HIVDataset_00001,0
HIVDataset_00002,False,CC(=O)N1c2ccccc2Sc2c1ccc1ccccc21,HIVDataset_00002,0
HIVDataset_00003,False,Nc1ccc(C=Cc2ccc(N)cc2S(=O)(=O)O)c(S(=O)(=O)O)c1,HIVDataset_00003,0
HIVDataset_00004,False,O=S(=O)(O)CCS(=O)(=O)O,HIVDataset_00004,0
...,...,...,...,...
HIVDataset_40665,False,CCC1CCC2c3c([nH]c4ccc(C)cc34)C3C(=O)N(N(C)C)C(...,HIVDataset_40665,0
HIVDataset_40666,False,Cc1ccc2[nH]c3c(c2c1)C1CCC(C(C)(C)C)CC1C1C(=O)N...,HIVDataset_40666,0
HIVDataset_40667,False,Cc1ccc(N2C(=O)C3c4[nH]c5ccccc5c4C4CCC(C(C)(C)C...,HIVDataset_40667,0


In [4]:
dataset.prepareDataset(
    split=RandomSplit(test_fraction=0.2, dataset=dataset),
    feature_calculators=[MorganFP(radius=2, nBits=2048)],
    recalculate_features=True,
)

In [5]:
from qsprpred.data.descriptors.sets import RDKitDescs

rdkit_descs = RDKitDescs()

dataset.addDescriptors([rdkit_descs])

dataset.descriptorSets

/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages/pandas/core/dtypes/astype.py:133: RuntimeWarning: overflow encountered in cast
  return arr.astype(dtype, copy=True)


In [6]:

# Přidejte cestu k vašemu lokálnímu repozitáři
import sys
import os

# Přidání cesty k lokálnímu repozitáři na začátek sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Zkontrolujte, zda je cesta v sys.path
print(sys.path)

from importlib import reload

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected
# Znovu načtěte modul, abyste zajistili, že je správně importován
reload(sys.modules['qsprpred.extra.gpu.models.neural_network'])

# Znovu proveďte import
from qsprpred.extra.gpu.models.neural_network import STFullyConnected

os.chdir('/home/ubuntu/Bakalarka/QSPRpred')
print(os.getcwd())


import sys
import importlib.util

# Přidání cesty k repozitáři do sys.path
sys.path.insert(0, '/home/ubuntu/Bakalarka/QSPRpred')

# Specifikujte cestu k souboru, který chcete importovat
module_path = '/home/ubuntu/Bakalarka/QSPRpred/qsprpred/extra/gpu/models/neural_network.py'
module_name = 'qsprpred.extra.gpu.models.neural_network'

# Načtěte modul z konkrétní cesty
spec = importlib.util.spec_from_file_location(module_name, module_path)
neural_network = importlib.util.module_from_spec(spec)
spec.loader.exec_module(neural_network)

# Nyní můžete používat třídu STFullyConnected
STFullyConnected = neural_network.STFullyConnected

['/home/ubuntu/Bakalarka/QSPRpred', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python311.zip', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/lib-dynload', '', '/home/ubuntu/miniconda3/envs/bakalarka_env/lib/python3.11/site-packages']
/home/ubuntu/Bakalarka/QSPRpred
lol


In [7]:
from sklearn.model_selection import ParameterGrid
from torch.nn import functional as F
from sklearn.metrics import f1_score
from sklearn.metrics import accuracy_score
import pandas as pd
def test_fun(dic,  X_train, y_train, X_test, y_test) -> pd.DataFrame:
    param_grid_t = ParameterGrid(dic)
    i = 0
    val_f1_t = []
    val_acc_t = []
    param_len_t = len(param_grid_t)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device used:", device)
    for param in param_grid_t:
        i += 1
        print(i, '/', param_len_t)
        model_sts_t = STFullyConnected(n_dim=X_train.shape[1],  # počet vstupních neuronů (počet deskriptorů)
        n_class=1,  # regresní úloha (1 výstup)
        gpus=[],
        device=device,
        is_reg=False, **param)
        model_sts_t.fit(X_train, y_train)
        res = model_sts_t.predict(X_test)
        res = res >0.5
        val_f1_t.append(f1_score(res, y_test))
        val_acc_t.append(accuracy_score(res, y_test))
        print(param)
        print(f1_score(res, y_test))
        print(accuracy_score(res, y_test))
    my_df = pd.DataFrame(param_grid_t)
    my_df["F1"] = val_f1_t
    my_df["Acc"] = val_acc_t
    return my_df

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()


X1 = scaler.fit_transform(dataset.X)
X2 = scaler.transform(dataset.X_ind)

In [10]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(sampling_strategy=0.5, random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X1, dataset.y)
display(pd.DataFrame(X_train_resampled))


,0,1,2,3,4,5,6,7,8,9,...,2248,2249,2250,2251,2252,2253,2254,2255,2256,2257
0,-0.127444,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,2.046839,-0.166249,-1.968275
1,-0.127444,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,-0.157540,-0.166249,1.446537
2,-0.127444,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,-0.157540,-0.166249,-0.669733
3,-0.127444,2.354537,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,-0.157540,-0.166249,-1.996689
4,-0.127444,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,6.384091,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,-0.157540,-0.166249,0.772795
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47101,-0.127444,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,-0.157540,-0.166249,0.742061
47102,-0.127444,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,4.251217,-0.166249,-1.195686
47103,-0.127444,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,-0.157540,-0.166249,-1.209790
47104,3.422601,-0.424712,-0.130034,-0.108997,-0.150112,-0.197461,-0.094005,-0.11213,-0.095327,-0.133892,...,-0.177221,-0.123609,-0.053491,-0.053134,-0.160251,-0.022825,-0.134943,-0.157540,4.375371,-0.884390


In [11]:
test_par = {'weight_decay': 0.0001, 'patience': 50, 
            'neuron_layers': [200],
            'n_epochs': 300, 'dropout_frac': 0.4,
            'act_fun': F.selu}
model_sts_3 = STFullyConnected(n_dim=X1.shape[1],  # počet vstupních neuronů (počet deskriptorů)
    n_class=1,  # regresní úloha (1 výstup)
    gpus=[],
    device="cuda",
    batch_size=256,is_reg=False, **test_par)
model_sts_3.fit(X_train_resampled, y_train_resampled)
res = model_sts_3.predict(X2)
res = res >0.5
print(f1_score(res, dataset.y_ind))

0.2642599277978339


In [12]:
from sklearn.metrics import accuracy_score
print(accuracy_score(res, dataset.y_ind))

0.8747233833292353


In [13]:
from torch import optim
import torch
my_dict_ult = {
    "act_fun": [F.selu],
    "dropout_frac": [0, 0.25, 0.5],
    "patience": [75],
    "tol": [1e-5],
    "weight_decay": [1e-4],
    "n_epochs": [200, 400],
    "neuron_layers": [[ 5000, 2500], [ 1024, 512, 256, 128, 64, 32, 16, 8, 4],  [2048, 1024, 512, 256, 128], [2256], ]
    , "batch_size": [256]
    , "optimizer": [optim.AdamW] 
}
df_batch_ult = test_fun(my_dict_ult, X_train_resampled, y_train_resampled, X2, dataset.y_ind)

Device used: cuda
1 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 200, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.34054834054834054
0.9438160806491271
2 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 200, 'neuron_layers': [1024, 512, 256, 128, 64, 32, 16, 8, 4], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.3583959899749373
0.9370543398082124
3 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 200, 'neuron_layers': [2048, 1024, 512, 256, 128], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.3244047619047619
0.9441849028768133
4 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epoc

In [14]:
df_batch_ult.sort_values(by="F1", ascending=False)

,act_fun,batch_size,dropout_frac,n_epochs,neuron_layers,optimizer,patience,tol,weight_decay,F1,Acc
17,<function selu at 0x7f90c22cd760>,256,0.50,200,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.399445,0.946767
23,<function selu at 0x7f90c22cd760>,256,0.50,400,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.386207,0.945291
7,<function selu at 0x7f90c22cd760>,256,0.00,400,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.386207,0.945291
15,<function selu at 0x7f90c22cd760>,256,0.25,400,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.386207,0.945291
9,<function selu at 0x7f90c22cd760>,256,0.25,200,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.378976,0.944800
13,<function selu at 0x7f90c22cd760>,256,0.25,400,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.362162,0.941972
21,<function selu at 0x7f90c22cd760>,256,0.50,400,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.361149,0.942587
12,<function selu at 0x7f90c22cd760>,256,0.25,400,"[5000, 2500]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.360248,0.949348
11,<function selu at 0x7f90c22cd760>,256,0.25,200,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.360144,0.934473
3,<function selu at 0x7f90c22cd760>,256,0.00,200,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.360144,0.934473


In [15]:
#{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 400, 'neuron_layers': [2256],
# 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
from torch import optim
import torch
my_dict_ult_2 = {
    "act_fun": [F.selu],
    "dropout_frac": [0, 0.25, 0.5],
    "patience": [75],
    "tol": [1e-5],
    "weight_decay": [1e-4],
    "n_epochs": [200, 150, 300],
    "neuron_layers": [[ 1024, 512, 256, 128, 64, 32, 16, 8, 4], [2256], [5000] ]
    , "batch_size": [256]
    , "optimizer": [optim.AdamW] 
}
df_batch_ult_2 = test_fun(my_dict_ult, X_train_resampled, y_train_resampled, X2, dataset.y_ind)

Device used: cuda
1 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 200, 'neuron_layers': [5000, 2500], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.34054834054834054
0.9438160806491271
2 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 200, 'neuron_layers': [1024, 512, 256, 128, 64, 32, 16, 8, 4], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.3583959899749373
0.9370543398082124
3 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 200, 'neuron_layers': [2048, 1024, 512, 256, 128], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
0.3244047619047619
0.9441849028768133
4 / 24
{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epoc

In [16]:
#{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0, 'n_epochs': 400, 'neuron_layers': [2256], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
#0.38620689655172413
#0.9452913695598721
#{'act_fun': <function selu at 0x7f90c22cd760>, 'batch_size': 256, 'dropout_frac': 0.5, 'n_epochs': 200, 'neuron_layers': [1024, 512, 256, 128, 64, 32, 16, 8, 4], 'optimizer': <class 'torch.optim.adamw.AdamW'>, 'patience': 75, 'tol': 1e-05, 'weight_decay': 0.0001}
#0.39944521497919555
#0.9467666584706171
df_batch_ult_2.sort_values(by="F1", ascending=False)

,act_fun,batch_size,dropout_frac,n_epochs,neuron_layers,optimizer,patience,tol,weight_decay,F1,Acc
17,<function selu at 0x7f90c22cd760>,256,0.50,200,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.399445,0.946767
23,<function selu at 0x7f90c22cd760>,256,0.50,400,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.386207,0.945291
7,<function selu at 0x7f90c22cd760>,256,0.00,400,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.386207,0.945291
15,<function selu at 0x7f90c22cd760>,256,0.25,400,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.386207,0.945291
9,<function selu at 0x7f90c22cd760>,256,0.25,200,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.378976,0.944800
13,<function selu at 0x7f90c22cd760>,256,0.25,400,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.362162,0.941972
21,<function selu at 0x7f90c22cd760>,256,0.50,400,"[1024, 512, 256, 128, 64, 32, 16, 8, 4]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.361149,0.942587
12,<function selu at 0x7f90c22cd760>,256,0.25,400,"[5000, 2500]",<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.360248,0.949348
11,<function selu at 0x7f90c22cd760>,256,0.25,200,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.360144,0.934473
3,<function selu at 0x7f90c22cd760>,256,0.00,200,[2256],<class 'torch.optim.adamw.AdamW'>,75,0.00001,0.0001,0.360144,0.934473
